# 04. 资源模型

资源模型是 SimPy 最常用的建模能力之一。它用于表达有限资源、连续库存、离散对象队列、优先级和抢占。

## 资源类型总览

| 类型 | 适合建模 | 核心操作 |
| --- | --- | --- |
| `Resource` | 同质有限资源，如柜台、机器、CPU 核 | `request()` / `release()` |
| `PriorityResource` | 带优先级的有限资源 | `request(priority=...)` |
| `PreemptiveResource` | 可抢占的有限资源 | `request(priority=..., preempt=True)` |
| `Container` | 连续数量，如油量、电量、库存量 | `put(amount)` / `get(amount)` |
| `Store` | 离散对象队列，如任务、消息、包裹 | `put(item)` / `get()` |
| `PriorityStore` | 带优先级的对象队列 | `put(PriorityItem(...))` / `get()` |
| `FilterStore` | 按条件取对象的队列 | `get(filter=...)` |


## Resource：普通共享资源

`Resource` 表示容量有限的同质资源。

In [3]:
import simpy


def user(env, name, resource):
    print(f"{env.now}: {name} 请求资源")
    with resource.request() as req:
        yield req
        print(f"{env.now}: {name} 获得资源")
        yield env.timeout(3)
        print(f"{env.now}: {name} 释放资源")


env = simpy.Environment()
resource = simpy.Resource(env, capacity=1)

env.process(user(env, "A", resource))
env.process(user(env, "B", resource))
env.run()

0: A 请求资源
0: B 请求资源
0: A 获得资源
3: A 释放资源
3: B 获得资源
6: B 释放资源


关键点：

- `capacity=1` 表示一次只能有一个使用者。
- `request()` 返回请求事件。
- `yield req` 等待请求成功。
- `with` 块结束时自动释放资源。

### Resource 的状态

常用属性：

```python
resource.capacity  # 容量
resource.count     # 当前正在使用的数量
resource.users     # 当前使用者请求对象列表
resource.queue     # 等待队列
```

可以用于监控：

```python
print(len(resource.queue), resource.count)
```

### 手动释放

不使用 `with` 时，需要手动释放：

```python
req = resource.request()
yield req
yield env.timeout(3)
resource.release(req)
```

推荐优先使用 `with`，减少异常和中断导致资源泄漏的风险。

## PriorityResource：优先级资源

`PriorityResource` 允许请求带优先级。数字越小，优先级越高。

In [4]:
import simpy


def user(env, name, resource, priority):
    with resource.request(priority=priority) as req:
        yield req
        print(f"{env.now}: {name} 获得资源，priority={priority}")
        yield env.timeout(2)


def blocker(env, resource):
    with resource.request(priority=0) as req:
        yield req
        yield env.timeout(1)


env = simpy.Environment()
resource = simpy.PriorityResource(env, capacity=1)

env.process(blocker(env, resource))
env.process(user(env, "low", resource, priority=5))
env.process(user(env, "high", resource, priority=1))
env.run()

1: high 获得资源，priority=1
3: low 获得资源，priority=5


优先级只影响等待队列顺序。已经获得资源的低优先级进程不会被普通 `PriorityResource` 抢占。


## PreemptiveResource：抢占资源

`PreemptiveResource` 支持高优先级请求抢占低优先级使用者。


In [9]:
import simpy


def job(env, name, resource, priority, arrive, work_time):
    yield env.timeout(arrive)
    print(f"{env.now}: {name} 到达")
    
    try:
        with resource.request(priority=priority, preempt=True) as req:
            yield req
            print(f"{env.now}: {name} 开始执行")
            yield env.timeout(work_time)
            print(f"{env.now}: {name} 正常完成")
    except simpy.Interrupt as interrupt:
        print(f"{env.now}: {name} 被抢占，原因: {interrupt.cause}")


env = simpy.Environment()
cpu = simpy.PreemptiveResource(env, capacity=1)

lowStart = 0
highStart = 3

for i in range(3):
    env.process(job(env, f"low_{i}", cpu, priority=5, arrive=lowStart+i, work_time=10))
    env.process(job(env, f"high_{i}", cpu, priority=1, arrive=highStart+i, work_time=2))
env.run()

0: low_0 到达
0: low_0 开始执行
1: low_1 到达
2: low_2 到达
3: high_0 到达
3: low_0 被抢占，原因: <simpy.resources.resource.Preempted object at 0x00000210F9A49A70>
3: high_0 开始执行
4: high_1 到达
5: high_2 到达
5: high_0 正常完成
5: high_1 开始执行
7: high_1 正常完成
7: high_2 开始执行
9: high_2 正常完成
9: low_1 开始执行
19: low_1 正常完成
19: low_2 开始执行
29: low_2 正常完成


抢占发生时，被抢占进程会收到 `simpy.Interrupt`。实际工程中通常需要记录已经执行的时间，然后决定是否重试剩余工作。


### 带剩余时间的抢占任务
```python
def preemptible_job(env, name, cpu, priority, work):
    remaining = work
    while remaining > 0:
        with cpu.request(priority=priority, preempt=True) as req:
            try:
                yield req
                start = env.now
                yield env.timeout(remaining)
                remaining = 0
                print(f"{env.now}: {name} 完成")
            except simpy.Interrupt:
                used = env.now - start
                remaining -= used
                print(f"{env.now}: {name} 被抢占，剩余 {remaining}")
```

这个模式适合 CPU 调度、抢占式服务、高优先级告警任务等。

## Container：连续容量

`Container` 用于建模连续数量。

In [10]:
import simpy


def consumer(env, tank):
    while True:
        yield tank.get(10)
        print(f"{env.now}: 消耗 10，剩余 {tank.level}")
        yield env.timeout(2)


def producer(env, tank):
    while True:
        yield env.timeout(5)
        yield tank.put(30)
        print(f"{env.now}: 补充 30，当前 {tank.level}")


env = simpy.Environment()
tank = simpy.Container(env, capacity=100, init=50)
env.process(consumer(env, tank))
env.process(producer(env, tank))
env.run(until=20)

0: 消耗 10，剩余 40
2: 消耗 10，剩余 30
4: 消耗 10，剩余 20
5: 补充 30，当前 50
6: 消耗 10，剩余 40
8: 消耗 10，剩余 30
10: 补充 30，当前 50
10: 消耗 10，剩余 50
12: 消耗 10，剩余 40
14: 消耗 10，剩余 30
15: 补充 30，当前 60
16: 消耗 10，剩余 50
18: 消耗 10，剩余 40


关键属性：

```python
tank.capacity  # 最大容量
tank.level     # 当前数量
```

`get(amount)` 会在库存不足时等待；`put(amount)` 会在容量不足时等待。

适合建模：

- 油箱和燃料。
- 电池和电量。
- 内存容量。
- 缓冲区容量。
- 原材料库存。

## Store：离散对象队列

`Store` 存储 Python 对象，按先进先出顺序取出。


In [11]:
import simpy


def producer(env, store):
    for i in range(3):
        item = f"item-{i}"
        yield store.put(item)
        print(f"{env.now}: 放入 {item}")
        yield env.timeout(1)


def consumer(env, store):
    while True:
        item = yield store.get()
        print(f"{env.now}: 取出 {item}")
        yield env.timeout(2)


env = simpy.Environment()
store = simpy.Store(env, capacity=2)
env.process(producer(env, store))
env.process(consumer(env, store))
env.run(until=10)

0: 放入 item-0
0: 取出 item-0
1: 放入 item-1
2: 取出 item-1
2: 放入 item-2
4: 取出 item-2


`capacity` 限制可存放对象数量。如果队列满了，`put()` 会等待；如果队列为空，`get()` 会等待。

适合建模：

- 请求队列。
- 消息队列。
- 任务池。
- 仓库货物。
- 待处理事件。

## PriorityStore：优先级对象队列

`PriorityStore` 按对象排序取出。官方推荐使用 `PriorityItem` 包装优先级和真实对象。


In [12]:
import simpy
from simpy.resources.store import PriorityItem


def demo(env, store):
    yield store.put(PriorityItem(3, "low"))
    yield store.put(PriorityItem(1, "high"))

    first = yield store.get()
    print(first.item)


env = simpy.Environment()
store = simpy.PriorityStore(env)
env.process(demo(env, store))
env.run()

high


数字越小越先取出。适合任务优先级、调度队列、事件优先级。


## FilterStore：按条件取对象

`FilterStore` 允许消费者按条件取对象。

In [13]:
import simpy


def consumer(env, store, target_type):
    item = yield store.get(lambda x: x["type"] == target_type)
    print(f"{env.now}: 取到 {item}")


def producer(env, store):
    yield store.put({"type": "cpu", "id": 1})
    yield store.put({"type": "gpu", "id": 2})


env = simpy.Environment()
store = simpy.FilterStore(env)
env.process(consumer(env, store, "gpu"))
env.process(producer(env, store))
env.run()

0: 取到 {'type': 'gpu', 'id': 2}


适合建模：

- 按机型选择服务器。
- 按函数类型选择实例。
- 按订单类型选择商品。
- 按目的地匹配车辆。

注意：`FilterStore` 的取出顺序不是简单 FIFO，而是取第一个满足过滤条件的对象。

## 资源监控

SimPy 不自动记录资源时间序列。可以手动采样：
```python
def monitor(env, resource, samples, interval=1):
    while True:
        samples.append({
            "time": env.now,
            "count": resource.count,
            "queue": len(resource.queue),
        })
        yield env.timeout(interval)
```

也可以在每次请求和释放时记录：

```python
def observe(env, resource, samples):
    samples.append((env.now, resource.count, len(resource.queue)))
```

采样适合画曲线；事件打点适合精确计算利用率和队列变化。

## 利用率计算

资源利用率可以通过面积法计算：

```python
def area_under_usage(samples):
    area = 0
    for left, right in zip(samples, samples[1:]):
        t0, usage0 = left
        t1, _ = right
        area += usage0 * (t1 - t0)
    return area
```

如果资源容量是 `capacity`，仿真总时长是 `T`：

```python
utilization = usage_area / (capacity * T)
```

对于严谨统计，推荐在每次资源状态变化时记录样本，而不是固定间隔采样。

## 选择哪种资源

| 建模问题 | 推荐类型 |
| --- | --- |
| N 个同质服务器 | `Resource(capacity=N)` |
| 高优先级任务先排队 | `PriorityResource` |
| 高优先级任务能打断低优先级任务 | `PreemptiveResource` |
| 油、电、内存、库存数量 | `Container` |
| 请求、消息、任务对象 | `Store` |
| 按优先级取任务对象 | `PriorityStore` |
| 按属性匹配对象 | `FilterStore` |

## 常见错误

### 把 Resource 当成对象池

`Resource` 只表示“有几个同质名额”，不保存具体对象。如果你需要取出具体对象，例如某台机器有自己的状态，应使用 `Store` 或自己维护对象列表。

### 忘记释放资源

错误：

```python
req = resource.request()
yield req
yield env.timeout(5)
```

正确：

```python
with resource.request() as req:
    yield req
    yield env.timeout(5)
```

### 用 Container 存离散对象

`Container` 只有数量，没有对象身份。如果需要知道取出的是哪个对象，用 `Store`。

### 抢占后不处理剩余工作

`PreemptiveResource` 只负责发出中断，不会自动帮你恢复任务。剩余工作、重试策略、失败统计都需要模型自己写。

## 小结

资源模型是把现实系统映射到 SimPy 的关键：

- 有限服务能力用 `Resource`。
- 排队优先级用 `PriorityResource`。
- 可打断服务用 `PreemptiveResource`。
- 连续库存用 `Container`。
- 离散对象流用 `Store`。
- 复杂匹配用 `FilterStore`。

资源不是单纯的数据结构，而是一组会产生事件的对象。请求、释放、放入、取出都会影响进程调度。